# SNA-MXH — Top-K Evaluation Audit (E1 + E3-A)
**Group 14 | Run-ready Google Colab**

Notebook kiểm tra **13 exports** (Most Popular 1, LightGCN 3, RLMRec-Con Full 3, User-only 3, Item-only 3).

**Cách dùng:** Mở notebook trên Google Colab → **Runtime > Run all** → cấp quyền Google Drive. Không cần train lại.

- Tự tìm `trn_mat.pkl` trong thư mục dự án; nếu không tìm thấy, điền `TRAIN_PATH` ở cell **CONFIG** rồi chạy lại.
- Kiểm tra shape, user/item IDs, duplicates, score order, cùng test users, train collision, metadata.
- Lưu `topk_audit_13_runs.csv` và `.json` vào `Data_SNA_Group14/experiments/audit_topk_20261029/`.
- `PASS_EXPORT_AUDIT` xác nhận exports và train mask; **không** thay thế việc đọc source code của ranking pipeline.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# -*- coding: utf-8 -*-
"""SNA-MXH / Group 14: independent E1+E3A Top-K export audit.

GOOGLE COLAB:
1) Mount Drive: from google.colab import drive; drive.mount('/content/drive')
2) Run this file or, preferably, open the companion RUNREADY.ipynb notebook.
3) No retraining. No existing data/checkpoints are modified.

Scope: 13 Top-20 exports = Most Popular (1) + LightGCN (3)
    + RLMRec-Con Full (3) + User-only (3) + Item-only (3).
Verifies shapes, IDs, duplicates, score ranking, matching test-user arrays,
original TRAIN interaction exclusion, and declared export metadata.

Important: Checking the NPZ and metadata does NOT prove the source evaluation
code never used validation/test labels to rank; source-code review is separate.
"""
from __future__ import annotations

import csv
import json
import os
import pickle
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
from scipy import sparse

PROJECT_ROOT = Path('/content/drive/MyDrive/Data_SNA_Group14')
TRAIN_PATH = None   # If autodiscovery fails: r'/content/drive/MyDrive/.../trn_mat.pkl'
OUTPUT_SUBDIR = 'experiments/audit_topk_20261029'
EXPECTED_SHAPE = (11000, 9332)  # users, items
EXPECTED_TEST_USERS = 9860
K = 20
SEEDS = (2023, 2024, 2025)
TOL_SCORE = 1e-6


def expected_runs(root: Path):
    pred = root / 'experiments' / 'predictions'
    yield ('Most Popular', '-', pred / 'most_popular' / 'topk_test.npz', 'most_popular')
    for model, dirname in (('LightGCN', 'lightgcn'), ('RLMRec-Con Full', 'lightgcn_plus')):
        for seed in SEEDS:
            yield (model, seed, pred / dirname / f'seed_{seed}' / 'topk_test.npz', dirname)
    e3 = root / 'experiments' / 'E3' / 'runs'
    for mode in ('user_only', 'item_only'):
        for seed in SEEDS:
            yield (f'E3 {mode}', seed,
                   e3 / mode / 'weight_e1_default' / f'seed_{seed}' / 'topk' / 'topk_test.npz',
                   'lightgcn_plus')


def find_training_matrix(root: Path, explicit_path=None):
    if explicit_path:
        p = Path(explicit_path).expanduser()
        if not p.is_file():
            print(f'[TRAIN] Configured TRAIN_PATH does not exist: {p}')
            return None, None, ['configured TRAIN_PATH does not exist']
        candidates = [p]
    else:
        candidates = []
        guesses = [
            root / 'datasets/amazon/trn_mat.pkl',
            root / 'data/amazon/trn_mat.pkl',
            root / 'dataset/amazon/trn_mat.pkl',
            root / 'RLMRec/datasets/amazon/trn_mat.pkl',
            root / 'RLMRec/data/amazon/trn_mat.pkl',
            root / 'RLMRec/dataset/amazon/trn_mat.pkl',
            root / 'trn_mat.pkl',
        ]
        for p in guesses:
            if p.is_file() and p not in candidates:
                candidates.append(p)
        # Limited recursive search only within project folder. Skip outputs/checkpoints.
        excluded = {'.git', '.ipynb_checkpoints', 'checkpoints', 'logs',
                    'experiments', 'predictions', '__pycache__', 'node_modules'}
        for directory, dirs, files in os.walk(root):
            rel = Path(directory).relative_to(root)
            if len(rel.parts) >= 7:
                dirs[:] = []
            else:
                dirs[:] = [d for d in dirs if d not in excluded]
            if 'trn_mat.pkl' in files:
                p = Path(directory) / 'trn_mat.pkl'
                if p not in candidates:
                    candidates.append(p)
    if not candidates:
        return None, None, ['trn_mat.pkl not found in PROJECT_ROOT; set TRAIN_PATH explicitly']

    errors = []
    for p in sorted(candidates, key=lambda x: ('amazon' not in str(x).lower(), len(str(x)))):
        try:
            # Only unpickle files in your own trusted Google Drive.
            with p.open('rb') as f:
                raw = pickle.load(f)
            if sparse.issparse(raw):
                mat = raw.tocsr(copy=True)
            elif isinstance(raw, np.ndarray) and raw.ndim == 2:
                mat = sparse.csr_matrix(raw)
            else:
                raise TypeError(f'Unsupported train matrix type: {type(raw).__name__}')
            if mat.shape != EXPECTED_SHAPE:
                raise ValueError(f'matrix shape {mat.shape}, expected {EXPECTED_SHAPE}')
            # Binary interactions: nonzero => observed in train.
            mat.eliminate_zeros()
            mat.data = np.ones(mat.nnz, dtype=np.bool_)
            mat.sum_duplicates()
            mat.sort_indices()
            print(f'[TRAIN] {p}')
            print(f'        shape={mat.shape}, train_interactions={mat.nnz:,}')
            return p, mat, errors
        except Exception as exc:
            errors.append(f'{p}: {type(exc).__name__} {exc}')
    return None, None, errors


def check_metadata(file_path: Path, expected_model: str, seed):
    """Check declared export metadata. This is not a source-code audit."""
    records = []
    errors, warnings = [], []
    meta_locations = [file_path.parent / 'export_meta.json',
                      file_path.parent / 'export_verification.json']
    found = 0
    for p in meta_locations:
        if not p.is_file():
            continue
        found += 1
        try:
            info = json.loads(p.read_text(encoding='utf-8'))
            records.append(str(p))
            model = info.get('model')
            if model and model != expected_model:
                errors.append(f'{p.name}: model={model}, expected {expected_model}')
            actual_seed = info.get('seed')
            if isinstance(seed, int) and actual_seed is not None and int(actual_seed) != seed:
                errors.append(f'{p.name}: seed={actual_seed}, expected {seed}')
            for key in ('ranking_mask',):
                if key in info and info[key] != 'train_only':
                    errors.append(f'{p.name}: {key}={info[key]!r}')
            if 'masked_splits' in info and set(info['masked_splits']) != {'train'}:
                errors.append(f'{p.name}: masked_splits={info["masked_splits"]!r}')
            if 'validation_masked' in info and info['validation_masked'] is not False:
                errors.append(f'{p.name}: validation_masked not false')
            for key in ('test_labels_used_to_rank', 'test_labels_used_for_ranking'):
                if key in info and info[key] is not False:
                    errors.append(f'{p.name}: {key} is not false')
            if 'n_test_users' in info and info['n_test_users'] != EXPECTED_TEST_USERS:
                errors.append(f'{p.name}: n_test_users differs')
            if 'test_user_count' in info and info['test_user_count'] != EXPECTED_TEST_USERS:
                errors.append(f'{p.name}: test_user_count differs')
            if 'n_items' in info and info['n_items'] != EXPECTED_SHAPE[1]:
                errors.append(f'{p.name}: n_items differs')
            if 'total_items' in info and info['total_items'] != EXPECTED_SHAPE[1]:
                errors.append(f'{p.name}: total_items differs')
            if 'k' in info and info['k'] != K:
                errors.append(f'{p.name}: k differs')
            if 'top_k' in info and info['top_k'] != K:
                errors.append(f'{p.name}: top_k differs')
            if 'status' in info and info['status'] != 'passed':
                errors.append(f'{p.name}: verification status not passed: {info["status"]}')
        except Exception as exc:
            errors.append(f'{p.name}: invalid JSON ({type(exc).__name__}: {exc})')
    if found == 0:
        warnings.append('No export metadata / verification file (NPZ checks still apply)')
    return records, errors, warnings


def audit_one(model: str, seed, path: Path, expected_model: str, train, ref_users):
    d = dict(model=model, seed=str(seed), file=str(path), status='FAIL',
             n_users='', k='', item_min='', item_max='', collisions='',
             unique_users=False, same_test_user_set=False, same_user_order=False,
             valid_users=False, valid_items=False, no_duplicates=False,
             scores_finite=False, scores_descending=False, no_train_items=False,
             metadata_files=[], warnings=[], errors=[], collision_examples=[])
    if not path.is_file():
        d['errors'].append('topk_test.npz missing')
        return d, ref_users

    try:
        with np.load(path, allow_pickle=False) as z:
            missing = {'user_ids', 'item_ids', 'scores'} - set(z.files)
            if missing:
                d['errors'].append('Missing NPZ fields: ' + ', '.join(sorted(missing)))
                return d, ref_users
            users = z['user_ids']
            items = z['item_ids']
            scores = z['scores']
        if users.ndim != 1 or items.ndim != 2 or scores.ndim != 2:
            d['errors'].append(f'Wrong dimensions: {users.shape}, {items.shape}, {scores.shape}')
            return d, ref_users
        n, k = items.shape
        d['n_users'], d['k'] = int(n), int(k)
        if users.shape != (EXPECTED_TEST_USERS,) or items.shape != (EXPECTED_TEST_USERS, K) or scores.shape != items.shape:
            d['errors'].append(f'Wrong shapes: users={users.shape}, items={items.shape}, scores={scores.shape}')
            return d, ref_users
        if users.dtype.kind not in 'iu' or items.dtype.kind not in 'iu' or scores.dtype.kind not in 'fiu':
            d['errors'].append('NPZ dtypes unexpected: user_ids/item_ids must be integers; scores numeric')
            return d, ref_users

        d['item_min'] = int(items.min())
        d['item_max'] = int(items.max())
        d['unique_users'] = np.unique(users).size == EXPECTED_TEST_USERS
        d['valid_users'] = bool(np.all((users >= 0) & (users < EXPECTED_SHAPE[0])))
        d['valid_items'] = bool(np.all((items >= 0) & (items < EXPECTED_SHAPE[1])))
        d['no_duplicates'] = bool(np.all(np.diff(np.sort(items, axis=1), axis=1) != 0))
        d['scores_finite'] = bool(np.all(np.isfinite(scores)))
        d['scores_descending'] = bool(np.all(np.diff(scores, axis=1) <= TOL_SCORE))
        for label in ('unique_users', 'valid_users', 'valid_items', 'no_duplicates', 'scores_finite', 'scores_descending'):
            if not d[label]:
                d['errors'].append(label)

        if ref_users is None and d['unique_users'] and d['valid_users']:
            ref_users = users.copy()
        if ref_users is not None:
            d['same_test_user_set'] = bool(np.array_equal(np.sort(users), np.sort(ref_users)))
            d['same_user_order'] = bool(np.array_equal(users, ref_users))
        if not d['same_test_user_set']:
            d['errors'].append('test-user set inconsistent across runs')
        if not d['same_user_order']:
            d['warnings'].append('Test-user order differs; align results using user_ids')

        if train is None:
            d['warnings'].append('TRAIN MATRIX MISSING: could not independently verify train collisions')
            d['no_train_items'] = None
        elif d['valid_items'] and d['valid_users']:
            rr = np.repeat(users, K)
            cc = items.reshape(-1)
            hits = np.asarray(train[rr, cc]).ravel().astype(bool)
            d['collisions'] = int(np.count_nonzero(hits))
            d['no_train_items'] = (d['collisions'] == 0)
            if not d['no_train_items']:
                d['errors'].append(f'{d["collisions"]} recommendations overlap train interactions')
                bad = np.flatnonzero(hits)[:12]
                d['collision_examples'] = [{'user': int(rr[i]), 'item': int(cc[i])} for i in bad]
        else:
            d['errors'].append('Cannot check train collisions with invalid user/item IDs')
        docs, meta_errors, meta_warnings = check_metadata(path, expected_model, seed)
        d['metadata_files'] = docs
        d['errors'] += meta_errors
        d['warnings'] += meta_warnings
        d['status'] = ('FAIL' if d['errors'] else
                       'INCOMPLETE' if train is None else 'PASS')
    except Exception as exc:
        d['errors'].append(f'{type(exc).__name__}: {exc}')
    return d, ref_users


def run_audit(project_root=PROJECT_ROOT, train_path=TRAIN_PATH):
    root = Path(project_root)
    print('=' * 78)
    print('SNA-MXH | INDEPENDENT Top-K AUDIT: E1 + E3-A (13 exports)')
    print('Project root:', root)
    print('Protocol: full-ranking / train-only mask / K=20 / 9860 test users')
    print('=' * 78)
    if not root.is_dir():
        print('[ERROR] Project root not found. Mount Google Drive, then check PROJECT_ROOT.')
        return None
    train_source, train, train_errors = find_training_matrix(root, train_path)
    if train is None:
        print('[WARN] trn_mat.pkl unavailable; result will be INCOMPLETE, never full PASS.')
        for e in train_errors: print('  ', e)
        print('       Set TRAIN_PATH above to the actual original Amazon trn_mat.pkl and rerun.')
    print('\nMODEL                     SEED   RESULT        TRAIN-COLLISIONS')
    print('-' * 78)
    results = []
    ref_users = None
    for model, seed, f, expected_model in expected_runs(root):
        r, ref_users = audit_one(model, seed, f, expected_model, train, ref_users)
        results.append(r)
        print(f'{model:<25} {str(seed):<6} {r["status"]:<13} {str(r["collisions"]):<8}')
        if r['errors']:
            print('      ERRORS:', '; '.join(r['errors'][:4]))
        if r['warnings']:
            print('      NOTES: ', '; '.join(r['warnings'][:2]))

    out = root / OUTPUT_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    summary = {
        'project': 'SNA-MXH',
        'audited_utc': datetime.now(timezone.utc).isoformat(),
        'protocol': {'ranking_mask': 'train_only', 'validation_masked': False,
                     'test_labels_used_for_ranking': 'metadata assertion only; source review pending',
                     'k': K, 'expected_test_users': EXPECTED_TEST_USERS,
                     'train_shape': list(EXPECTED_SHAPE)},
        'train_source': str(train_source) if train_source else None,
        'train_discovery_errors': train_errors,
        'counts': {s: sum(x['status'] == s for x in results) for s in ('PASS', 'FAIL', 'INCOMPLETE')},
        'source_code_review': 'NOT PERFORMED',
        'runs': results,
    }
    n_pass = summary['counts']['PASS']
    summary['overall_status'] = ('PASS_EXPORT_AUDIT' if n_pass == 13 else
                                 'FAIL' if summary['counts']['FAIL'] else 'INCOMPLETE')
    json_path = out / 'topk_audit_13_runs.json'
    csv_path = out / 'topk_audit_13_runs.csv'
    json_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    columns = ['model', 'seed', 'status', 'n_users', 'k', 'item_min', 'item_max',
               'collisions', 'unique_users', 'same_test_user_set', 'same_user_order',
               'valid_users', 'valid_items', 'no_duplicates', 'scores_finite',
               'scores_descending', 'no_train_items', 'errors', 'warnings',
               'metadata_files', 'collision_examples', 'file']
    with csv_path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        for d in results:
            row = dict(d)
            for field in ('errors', 'warnings', 'metadata_files', 'collision_examples'):
                row[field] = json.dumps(row[field], ensure_ascii=False)
            writer.writerow({k: row.get(k) for k in columns})

    print('\n' + '=' * 78)
    print(f'OVERALL: {summary["overall_status"]} | PASS={n_pass}/13, FAIL={summary["counts"]["FAIL"]}, INCOMPLETE={summary["counts"]["INCOMPLETE"]}')
    print('CSV:', csv_path)
    print('JSON:', json_path)
    print('Note: independent source-code masking audit remains out of scope.')
    print('=' * 78)
    return summary



In [ ]:
# CHỈ CẦN SỬA NẾU FILE Ở VỊ TRÍ KHÁC
PROJECT_ROOT = '/content/drive/MyDrive/Data_SNA_Group14'
TRAIN_PATH = None  # VD: '/content/drive/MyDrive/Data_SNA_Group14/datasets/amazon/trn_mat.pkl'
print('Project:', PROJECT_ROOT)
print('Train override:', TRAIN_PATH or 'auto-discover')


In [ ]:
# CHẠY TẤT CẢ 13 EXPORTS, KHÔNG TRAIN LẠI
result = run_audit(project_root=PROJECT_ROOT, train_path=TRAIN_PATH)

if result is not None:
    try:
        import pandas as pd
        from IPython.display import display
        csv_file = Path(PROJECT_ROOT) / OUTPUT_SUBDIR / 'topk_audit_13_runs.csv'
        cols = ['model', 'seed', 'status', 'collisions', 'same_test_user_set',
                'valid_items', 'no_duplicates', 'scores_descending', 'no_train_items']
        display(pd.read_csv(csv_file)[cols])
    except ImportError:
        print('Pandas unavailable; CSV was saved successfully.')
